In [5]:
import tensorflow as tf

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os

folder_path = '/content/drive/MyDrive/numberplate/number plate'


print(f"Checking contents of: {folder_path}")
if os.path.exists(folder_path):
    contents = os.listdir(folder_path)
    if not contents:
        print(f"The directory '{folder_path}' is empty.")
    else:
        print(f"Contents of '{folder_path}': {contents}")
else:
    print(f"Error: The directory '{folder_path}' does not exist. Please ensure your Google Drive is mounted and the folder path is correct.")

Checking contents of: /content/drive/MyDrive/numberplate/number plate
Contents of '/content/drive/MyDrive/numberplate/number plate': ['annotations', 'images']


In [8]:
image_folder_path = "/content/drive/MyDrive/numberplate/number plate/images"

dataset = tf.keras.utils.image_dataset_from_directory(
    image_folder_path,
    labels=None,
    image_size=(224, 224),
    batch_size=32,
    shuffle=True
)


Found 473 files.


In [9]:
def preprocess_image(image, target_size):
    image = tf.image.resize(image, target_size)
    image = tf.cast(image, tf.float32) / 255.0
    return image

In [10]:
resized_dataset = dataset.map(lambda image: preprocess_image(image, target_size=(224, 224)))

In [11]:
dataset = tf.keras.utils.image_dataset_from_directory(
    image_folder_path,
    labels=None,
    image_size=(224, 224),
    batch_size=32,
    shuffle=False
)


Found 473 files.


In [68]:
import tensorflow as tf
import os
import pandas as pd
import xml.etree.ElementTree as ET

def normalize_bounding_box(box_coords, original_width, original_height):
    xmin_norm = box_coords['xmin'] / original_width
    ymin_norm = box_coords['ymin'] / original_height
    xmax_norm = box_coords['xmax'] / original_width
    ymax_norm = box_coords['ymax'] / original_height
    return [xmin_norm, ymin_norm, xmax_norm, ymax_norm]


annotations_folder_path = os.path.join(folder_path, 'annotations')
annotation_data = []

if os.path.exists(annotations_folder_path):
    for annotation_file in os.listdir(annotations_folder_path):
        if annotation_file.endswith('.xml'):
            xml_path = os.path.join(annotations_folder_path, annotation_file)
            tree = ET.parse(xml_path)
            root = tree.getroot()

            filename = root.find('filename').text
            width = int(root.find('size').find('width').text)
            height = int(root.find('size').find('height').text)

            boxes = []
            for obj in root.findall('object'):
                bndbox = obj.find('bndbox')
                xmin = int(bndbox.find('xmin').text)
                ymin = int(bndbox.find('ymin').text)
                xmax = int(bndbox.find('xmax').text)
                ymax = int(bndbox.find('ymax').text)
                boxes.append({'xmin': xmin, 'ymin': ymin, 'xmax': xmax, 'ymax': ymax})

            annotation_data.append({
                'filename': filename,
                'width': width,
                'height': height,
                'boxes': boxes
            })
    annotations_df = pd.DataFrame(annotation_data)
    print(f"Loaded {len(annotations_df)} annotations from {annotations_folder_path}.")
else:
    print(f"Error: Annotation folder '{annotations_folder_path}' does not exist.")
    annotations_df = pd.DataFrame()


dataset_items = []


if not annotations_df.empty:
    for index, row in annotations_df.iterrows():
        filename = row['filename']
        width = row['width']
        height = row['height']
        boxes = row['boxes']

        image_path = os.path.join(image_folder_path, filename)

        normalized_boxes_for_image = []
        for box_dict in boxes:
            normalized_coords = normalize_bounding_box(box_dict, width, height)
            normalized_boxes_for_image.append(normalized_coords)

        dataset_items.append((image_path, normalized_boxes_for_image))

    image_paths = [item[0] for item in dataset_items]
    all_normalized_boxes = [item[1] for item in dataset_items]


    normalized_bboxes_ragged = tf.ragged.constant(all_normalized_boxes, dtype=tf.float32)


    bbox_dataset = tf.data.Dataset.from_tensor_slices((image_paths, normalized_bboxes_ragged))

    print("Element spec of bbox_dataset:")
    print(bbox_dataset.element_spec)
else:
    print("No annotations were loaded, cannot create bbox_dataset.")

Loaded 433 annotations from /content/drive/MyDrive/numberplate/number plate/annotations.
Element spec of bbox_dataset:
(TensorSpec(shape=(), dtype=tf.string, name=None), RaggedTensorSpec(TensorShape([None, None]), tf.float32, 1, tf.int64))


In [59]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss=bbox_loss)

print("Model compiled successfully with Adam optimizer and custom bbox_loss.")

Model compiled successfully with Adam optimizer and custom bbox_loss.


In [60]:


@tf.function
def bbox_loss(y_true, y_pred):

    if not isinstance(y_true, tf.RaggedTensor):

        pass



    batch_losses = []
    for i in tf.range(tf.shape(y_pred)[0]):
        true_boxes_for_image = y_true[i]
        predicted_box_for_image = y_pred[i]

        if tf.equal(tf.shape(true_boxes_for_image)[0], 0):

            batch_losses.append(0.0)
        else:

            predicted_boxes_repeated = tf.tile(tf.expand_dims(predicted_box_for_image, axis=0), [tf.shape(true_boxes_for_image)[0], 1])


            mse = tf.reduce_mean(tf.square(true_boxes_for_image - predicted_boxes_repeated))
            batch_losses.append(mse)


    return tf.reduce_mean(tf.stack(batch_losses))

print("Custom bounding box loss function 'bbox_loss' defined and decorated with @tf.function.")

Custom bounding box loss function 'bbox_loss' defined and decorated with @tf.function.


In [61]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss=bbox_loss)

print("Model compiled successfully with Adam optimizer and custom bbox_loss.")

Model compiled successfully with Adam optimizer and custom bbox_loss.


In [62]:
import tensorflow as tf

@tf.function
def bbox_loss(y_true, y_pred):
    # y_true is now (batch_size, 4) from the modified dataset pipeline
    # y_pred is also (batch_size, 4) from the model output

    # Filter out padded ground truth boxes (marked with -1.0, indicating no true box for that image)
    valid_mask = tf.reduce_any(tf.not_equal(y_true, -1.0), axis=-1) # Shape (batch_size,)

    # Apply the mask to both y_true and y_pred to only consider valid examples
    filtered_y_true = tf.boolean_mask(y_true, valid_mask)
    filtered_y_pred = tf.boolean_mask(y_pred, valid_mask)

    # Calculate MSE only for valid boxes. If no valid boxes in batch, return 0 loss.
    if tf.equal(tf.shape(filtered_y_true)[0], 0):
        return tf.constant(0.0, dtype=tf.float32)
    else:
        mse = tf.reduce_mean(tf.square(filtered_y_true - filtered_y_pred))
        return mse

print("Custom bounding box loss function 'bbox_loss' redefined for single target bounding box.")

Custom bounding box loss function 'bbox_loss' redefined for single target bounding box.


In [63]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
model.compile(optimizer=optimizer, loss=bbox_loss)

print("Model compiled successfully with Adam optimizer and custom bbox_loss.")

Model compiled successfully with Adam optimizer and custom bbox_loss.


In [64]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # or your specific loss
    metrics=['accuracy'] # <--- Add this
)


In [ ]:
# 1. Make sure your model ends with Dense(4)
# 2. Compile with 'mae' as a proxy for accuracy
optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)

model.compile(
    optimizer=optimizer,
    loss='mse',      # Using built-in Mean Squared Error is safer
    metrics=['mae']  # Mean Absolute Error tells you how many pixels off you are
)

epochs = 20
history = model.fit(
    train_dataset,
    epochs=epochs,
    validation_data=val_dataset
)

# Output results
print("\n" + "="*30)
print(f"Final Training MAE: {history.history['mae'][-1]:.4f}")
print(f"Final Validation MAE: {history.history['val_mae'][-1]:.4f}")
print(f"Final Training Loss: {history.history['loss'][-1]:.4f}")
print("="*30)

Epoch 1/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 35s 2s/step - loss: 0.0575 - mae: 0.1916 - val_loss: 0.0254 - val_mae: 0.1202
Epoch 2/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - loss: 0.0238 - mae: 0.1177 - val_loss: 0.0214 - val_mae: 0.1101
Epoch 3/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - loss: 0.0187 - mae: 0.1005 - val_loss: 0.0123 - val_mae: 0.0793
Epoch 4/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - loss: 0.0125 - mae: 0.0819 - val_loss: 0.0132 - val_mae: 0.0803
Epoch 5/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - loss: 0.0113 - mae: 0.0748 - val_loss: 0.0087 - val_mae: 0.0665
Epoch 6/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - loss: 0.0110 - mae: 0.0724 - val_loss: 0.0075 - val_mae: 0.0604
Epoch 7/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - loss: 0.0086 - mae: 0.0653 - val_loss: 0.0080 - val_mae: 0.0603
Epoch 8/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 27s 2s/step - loss: 0.0077 - mae: 0.0618 - val_loss: 0.0061 - val_mae: 0.0559
Epoch 9/20
11/11 ━━━━━━━━━━━━━━━━━━━━ 41s 2s/step - loss: 0.0065 - mae: 

In [67]:
TARGET_IMAGE_SIZE = (224, 224)
MAX_BOXES_PER_IMAGE = 5 # Set a reasonable max number of boxes to pad to

def load_and_preprocess_image(image_path, normalized_bboxes):

    img = tf.io.read_file(image_path)

    img = tf.image.decode_image(img, channels=3, expand_animations=False)

    img = tf.image.resize(img, TARGET_IMAGE_SIZE)

    img = tf.cast(img, tf.float32) / 255.0

    # Convert RaggedTensor bboxes to dense tensor by padding
    # Explicitly convert to dense tensor first
    dense_bboxes = normalized_bboxes.to_tensor(default_value=-1.0)

    # Filter out padded boxes and take the first one
    valid_boxes_mask = tf.reduce_any(tf.not_equal(dense_bboxes, -1.0), axis=-1)
    filtered_true_boxes = tf.boolean_mask(dense_bboxes, valid_boxes_mask) # Shape (num_valid_boxes, 4)

    # Choose the first valid box if available, otherwise a placeholder
    single_target_bbox = tf.cond(
        tf.equal(tf.shape(filtered_true_boxes)[0], 0),
        lambda: tf.constant([-1.0, -1.0, -1.0, -1.0], dtype=tf.float32),
        lambda: filtered_true_boxes[0]
    )
    # Explicitly set the shape to ensure TensorFlow knows it's (4,)
    single_target_bbox = tf.ensure_shape(single_target_bbox, (4,))

    return img, single_target_bbox # single_target_bbox shape (4,)

# Explicitly define the output signature for the map function
output_signature = (
    tf.TensorSpec(shape=TARGET_IMAGE_SIZE + (3,), dtype=tf.float32),
    tf.TensorSpec(shape=(4,), dtype=tf.float32)
)

processed_dataset = bbox_dataset.map(
    load_and_preprocess_image,
    num_parallel_calls=tf.data.AUTOTUNE,
    deterministic=False,
    output_signature=output_signature
)


print("Element spec of processed_dataset:")
print(processed_dataset.element_spec)

TypeError: DatasetV2.map() got an unexpected keyword argument 'output_signature'

In [69]:
DATASET_SIZE = tf.data.experimental.cardinality(processed_dataset).numpy()
if DATASET_SIZE == tf.data.experimental.UNKNOWN_CARDINALITY:

    DATASET_SIZE = 0
    for _ in processed_dataset:
        DATASET_SIZE += 1

TRAIN_SPLIT_RATIO = 0.8
VAL_SPLIT_RATIO = 0.2

TRAIN_SIZE = int(TRAIN_SPLIT_RATIO * DATASET_SIZE)
VAL_SIZE = int(VAL_SPLIT_RATIO * DATASET_SIZE)

print(f"Total dataset size: {DATASET_SIZE}")
print(f"Training set size: {TRAIN_SIZE}")
print(f"Validation set size: {VAL_SIZE}")

SHUFFLE_BUFFER_SIZE = DATASET_SIZE
BATCH_SIZE = 32

shuffled_dataset = processed_dataset.shuffle(SHUFFLE_BUFFER_SIZE)


train_dataset = shuffled_dataset.take(TRAIN_SIZE)
val_dataset = shuffled_dataset.skip(TRAIN_SIZE).take(VAL_SIZE)


train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Train dataset element spec:", train_dataset.element_spec)
print("Validation dataset element spec:", val_dataset.element_spec)

for element in train_dataset.take(1):
    print("Shape of images in one train batch:", element[0].shape)
    print("Shape of bboxes in one train batch:", element[1].shape)

for element in val_dataset.take(1):
    print("Shape of images in one validation batch:", element[0].shape)
    print("Shape of bboxes in one validation batch:", element[1].shape)

Total dataset size: 433
Training set size: 346
Validation set size: 86
Train dataset element spec: (TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 4), dtype=tf.float32, name=None))
Validation dataset element spec: (TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None, 4), dtype=tf.float32, name=None))
Shape of images in one train batch: (32, 224, 224, 3)
Shape of bboxes in one train batch: (32, 4)
Shape of images in one validation batch: (32, 224, 224, 3)
Shape of bboxes in one validation batch: (32, 4)
